In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet


# Load
train = pd.read_csv('train.csv/train.csv') 
features = pd.read_csv('features.csv/features.csv')

# --- 1. Data Preparation & Aggregation ---
# (Assumes 'data' is already merged from train and features)
store_data = data[data['Store'] == 1].groupby('Date').agg({
    'Weekly_Sales': 'sum',
    'IsHoliday': 'max'
}).sort_index()

ts_data = store_data['Weekly_Sales'].resample('W').sum()
exog_data = store_data['IsHoliday'].resample('W').max().fillna(0)

train_size = int(len(ts_data) * 0.8)
train_ts, test_ts = ts_data[:train_size], ts_data[train_size:]
train_exog, test_exog = exog_data[:train_size], exog_data[train_size:]
forecast_steps = len(test_ts)

# --- 2. Statistical Models ---
# ARIMA
model_arima = ARIMA(train_ts, order=(1,1,1)).fit()
forecast_arima = model_arima.forecast(steps=forecast_steps)

# SARIMAX
model_sarimax = SARIMAX(train_ts, exog=train_exog, order=(1,1,1), 
                        seasonal_order=(1,1,1,52), enforce_stationarity=False).fit(disp=False)
forecast_sarimax = model_sarimax.get_forecast(steps=forecast_steps, exog=test_exog).predicted_mean

# Exponential Smoothing (Damped)
model_es = ExponentialSmoothing(train_ts, trend='add', seasonal='add', 
                                seasonal_periods=52, damped_trend=True).fit()
forecast_es = model_es.forecast(steps=forecast_steps)

# --- 3. Deep Learning Models (Fixed Scaling & Windows) ---
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_train = scaler.fit_transform(train_ts.values.reshape(-1, 1))
scaled_test = scaler.transform(test_ts.values.reshape(-1, 1))

def create_sequences(data, window_size=4):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 4
X_train, y_train = create_sequences(scaled_train, window_size)
X_test, y_test = create_sequences(scaled_test, window_size)
X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# LSTM Implementation
model_lstm = Sequential([
    LSTM(50, activation='relu', input_shape=(window_size, 1)),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_train, y_train, epochs=50, batch_size=4, verbose=0)

# CRITICAL FIX: Inverse Scaling
raw_pred_lstm = model_lstm.predict(X_test_reshaped)
forecast_lstm = scaler.inverse_transform(raw_pred_lstm).flatten()

# --- 4. Facebook Prophet ---
df_prophet = ts_data.reset_index().rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
model_prophet = Prophet(yearly_seasonality=True).fit(df_prophet.iloc[:train_size])
future = model_prophet.make_future_dataframe(periods=forecast_steps, freq='W')
forecast_prophet = model_prophet.predict(future)
prophet_pred = forecast_prophet['yhat'].iloc[train_size:].values

# --- 5. Final Evaluation & Visualization ---
def evaluate(actual, pred, name):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    return {'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2)}

# Align lengths for metrics (LSTM/RNN lose 'window_size' steps)
actual_trimmed = test_ts.values[window_size:]

results = [
    evaluate(test_ts, forecast_arima, 'ARIMA'),
    evaluate(test_ts, forecast_sarimax, 'SARIMAX'),
    evaluate(test_ts, forecast_es, 'Exp Smoothing'),
    evaluate(actual_trimmed, forecast_lstm, 'LSTM'),
    evaluate(test_ts, prophet_pred, 'Prophet')
]

print(pd.DataFrame(results))

# CRITICAL FIX: Datetime Alignment for Plotting
plt.figure(figsize=(15, 6))
plt.plot(test_ts.index, test_ts, label='Actual Sales', color='black', linewidth=2)
plt.plot(test_ts.index, forecast_es, label='Exp Smoothing (Damped)', color='red')
plt.plot(test_ts.index[window_size:], forecast_lstm, label='LSTM Forecast', color='purple')
plt.plot(test_ts.index, prophet_pred, label='Prophet Forecast', color='blue', linestyle='--')
plt.title('Corrected Walmart Sales Forecast Comparison')
plt.legend()
plt.show()

Importing plotly failed. Interactive plots will not work.


NameError: name 'data' is not defined